In [1]:
using Pkg
using LinearAlgebra
using Symbolics
using MultivariatePolynomials
using DynamicPolynomials
using HomotopyContinuation             
using Distributions
using Random
using Plots  
using PrettyTables

# Função que arredonda polinômios

function roundp(p)
      if p isa AbstractPolynomialLike
          cs = coefficients(p)
          ms = monomials(p)
          return isempty(cs) ? zero(p) : sum(round(c; digits=1) * m for (c, m) in zip(cs, ms))
      elseif p isa Number
          return round(p; digits=2)
      elseif p isa AbstractArray
          return map(roundp, p)
      else
          error("Unsupported type in roundp: $(typeof(p))")
      end
  end
  # -----

function expand_vector(v) 
    return [ModelKit.expand(v[i]) for i in 1:length(v)] end   

expand_vector (generic function with 1 method)

# Basic Specification

My goal is to solve


$$
\min_{\beta} \; Q( \beta    ) \quad \quad s.t.  \quad \sum_{i=1}^k \frac{p(\beta_i)}{q( \beta_i)} = 1
$$

Where 

$$
p(\beta_i) = \beta_i N ( \beta_i) -  \beta_i N ( -\beta_i) \quad \quad q(\beta_i) = N ( \beta_i) +  N ( -\beta_i)
$$
  
   The Lagrangean is

$$
\mathcal{L}
=
Q(\beta)
+
\lambda \left[ \sum_{i=1}^k \frac{p(\beta_i)}{q( \beta_i)}  - 1  \right] = 0 
$$

Differentiating wrt to $\beta_i$ 

$$
\tfrac{\partial}{\partial \beta_i} Q_n(\beta)
+
\lambda 
\sum_{i=1}^k \frac{q(\beta_i) p'(\beta_i) - p(\beta_i)q'(\beta_i)}{q(\beta_i)^2} = 0
$$

To turn this into a polinomial system, we must multiply it by $ Q_2( \beta) = \prod_{j=1}^k q(\beta_j)^2$. The system with the KT conditions is

$$
\begin{split}
& Q_2( \beta)  \big\{ \nabla Q_n(\beta)
+
\lambda \nabla R( \beta) \big\} = 0  \\

& Q_1( \beta)(R( \beta)  - 1) = 0 

\end{split}
$$

Where $Q_1( \beta) = \prod_{j=1}^k q(\beta_j)$. This is 

$$
\begin{split}
\tfrac{\partial}{\partial \beta_i} Q_n(\beta) \prod_{j=1}^k q(\beta_j)^2 + \lambda \sum q(\beta_i) p'(\beta_i) - p(\beta_i)q'(\beta_i) & = 0 \\
\sum_{i=1}^k p( \beta_i) \prod_{j \neq i} q ( \beta_j)  - \prod_{j=1}^k q( \beta_j) & = 0 
\end{split}
$$

We have $k+1$ variables and degree $2k(d-1)$.



# Performance

In [64]:
d_vals = 4:8
k_vals = 2:3

T = Array{Float64}(undef, length(d_vals), length(k_vals))   # matriz D×K



for d in d_vals, k in k_vals

    println( [ "d = $d" , "k = $k"   ])
    
    a = exp( - 1 / sqrt(d) ) 
     
    function N(b) 
        return prod(   [ b + a^k  for k in 1:d-1   ]  )   
    end
       
    function p_num(x)
        return x * ( N(x) - N(-x)) 
    end 
    
    function q_num(x)
        return  N(x) + N(-x)
    end     

    Random.seed!(126)

    X = rand(n, k)
    beta_0 = [i for i in 0.1:1:k-1 + 0.1]
    
    
    y = X* beta_0 + rand(Normal(0,1), n) ; 

    @var β[1:k] λ s

    function p_num(x)
        return x * ( N(x) - N(-x)) 
    end 
    
    function q_num(x)
        return  N(x) + N(-x)
    end 
    
    function Q(b)
        return  (y - X*b)'*(y - X*b)
    end 
    
    
    p = [  p_num(β[i])   for i in 1:k  ]  
    q = [  q_num(β[i])   for i in 1:k  ]  
    
    pQ = [    prod([ q[j] for j in 1:k if j ≠ i  ])   for i in 1:k  ]
    
    
    Dp = [    differentiate( p[i] , β[i]  )  for i in 1:k  ]
    Dq = [    differentiate( q[i] , β[i]  )  for i in 1:k  ]
    DQ = [    differentiate( Q(β) , β[i]  )  for i in 1:k  ]
    
    eqA = expand_vector(    [   q[i]^2*DQ[i] + λ*( q[i]* Dp[i] - p[i]*Dq[i]  )  for i in 1:k ]  ) 
    eqB = expand_vector(  sum( [  p[i]*pQ[i]  for i in 1:k ] ) - prod( q )    )
    
    F = vec( [eqA ; eqB ] ) 
    
    F_sys = System(F; variables=[ β ; λ  ])

    time = @elapsed result = solve(F_sys; start_system=:total_degree ,show_progress=false) 

    if count(u -> all(abs.(imag.(u)) .< tol ), complex_sols_1)  ≠ 0 println("Success")  end

    line = findfirst(==(d), d_vals)
    column = findfirst(==(k), k_vals)

    T[line,column] = time     # T[d,k]

end    

["d = 3", "k = 2"]
Sucess
["d = 3", "k = 3"]
Sucess
["d = 4", "k = 2"]
Sucess
["d = 4", "k = 3"]
Sucess
["d = 5", "k = 2"]
Sucess
["d = 5", "k = 3"]
Sucess
["d = 6", "k = 2"]
Sucess
["d = 6", "k = 3"]
Sucess
["d = 7", "k = 2"]
Sucess
["d = 7", "k = 3"]
Sucess
["d = 8", "k = 2"]
Sucess
["d = 8", "k = 3"]
Sucess


In [66]:
Head = ["d"; string.("k=", k_vals)]
data = hcat(d_vals, T)

pretty_table(data, header=Head)

┌─────┬───────────┬──────────┐
│   d │       k=2 │      k=3 │
├─────┼───────────┼──────────┤
│ 3.0 │ 0.0355994 │ 0.559944 │
│ 4.0 │  0.124798 │  1.87577 │
│ 5.0 │  0.547777 │  13.4401 │
│ 6.0 │   4.11135 │  28.2997 │
│ 7.0 │   5.77066 │  128.904 │
│ 8.0 │   7.75692 │  383.066 │
└─────┴───────────┴──────────┘


# Alternative specification

Introduce new slack variables $r_i$ for $i = 1, \cdots , k$ such that $\frac{p(\beta_i)}{q(\beta_i)}= r_i$

$$
\min_{\beta, r } \; 
Q(\beta) 
\quad s.t.
\begin{cases}
& \sum_{i=1}^k r_i = 1 \\
& p(\beta_i) - r_i q(\beta_i) = 0
\end{cases}
$$

The Lagrangean is

$$
\mathcal{L} = 
Q( \beta) + \lambda \left[ \sum_{i=1}^k r_i -1  \right] + outra restricao?
$$

The system has dimensions $3k +1$ and degree .

$$
\begin{split}
\tfrac{\partial}{\partial \beta } \mathcal{L} & = 0 \\
\tfrac{\partial}{\partial r } \mathcal{L}     & = 0 \\
\sum r_i -1 & = 0 \\
p(\beta_i) - r_i  q(\beta_i)^2 & = 0
\end{split}
$$

In [2]:
# inviável

n = 1000
d_vals = 6:8
k_vals = 2:3

T = Array{Float64}(undef, length(d_vals), length(k_vals))   # matriz D×K

for d in d_vals, k in k_vals

    println( [ "d = $d" , "k = $k"   ])
    
    a = exp( - 1 / sqrt(d) ) 
     
    function N(b) 
        return prod(   [ b + a^k  for k in 1:d-1   ]  )   
    end
        
    function p(x)
        return x * ( N(x) - N(-x)) 
    end 

    function q(x)
        return  N(x) + N(-x)
    end     

    Random.seed!(126)

    X = rand(n, k)
    beta_0 = [i for i in 0.1:1:k-1 + 0.1]


    y = X* beta_0 + rand(Normal(0,1), n) ; 

    @var β[1:k] λ r[1:k] μ[1:k]

    function p_num(x)
        return x * ( N(x) - N(-x)) 
    end 

    function q_num(x)
        return  N(x) + N(-x)
    end 

    function Q(b)
        return  (y - X*b)'*(y - X*b)
    end 


    L = Q(β) + λ * (sum( [r[i] for i in 1:k ] ) -1 ) 


    eqA = expand_vector(    [  differentiate( L , β[i])  for i in 1:k ]  ) 
    eqB = expand_vector(    [  differentiate( L , r[i])  for i in 1:k ]  ) 
    eqC = expand_vector(      sum([   r[i]  for i in 1:k ]) - 1  )
    eqD = expand_vector(    [  p(β[i])  - r[i]*q(β[i])^2   for i in 1:k ]  )

    F = vec( [eqA ; eqB ; eqC; eqD] ) 

    F_sys = System(F; variables=[ β ; r ; λ])

    time = @elapsed result = solve(F_sys; start_system=:total_degree ,show_progress=false) 

    tol = 10^(-6)

    if count(u -> all(abs.(imag.(u)) .< tol ), solutions(result))  == 0 println("Failure")  
    else println("Success")  
    end

    line = findfirst(==(d), d_vals)
    column = findfirst(==(k), k_vals)

    T[line,column] = time     # T[d,k]

end    



["d = 6", "k = 2"]
Failure
["d = 6", "k = 3"]
Failure
["d = 7", "k = 2"]
Failure
["d = 7", "k = 3"]
Failure
["d = 8", "k = 2"]
Failure
["d = 8", "k = 3"]
Failure


In [ ]:
Head = ["d"; string.("k=", k_vals)]
data = hcat(d_vals, T)

pretty_table(data, header=Head)

# Sum of Squares

We can eliminate the multipliers $\mu_i$ for $i = 2 , \cdots , k$ if we write the system as

$$
\min_{\beta, r } \; 
Q(\beta) 
\quad s.t.
\begin{cases}
& \sum_{i=1}^k r_i = 1 \\
& \sum_{i=1}^k \big[ p(\beta_i) - r_i q(\beta_i) \big]^2 = 0
\end{cases}
$$

So 

$$
\mathcal{L} = 
Q( \beta) + \lambda \left[ \sum_{i=1}^k r_i -1  \right]
+
\mu \sum_{i=1}^k  \Big[ p(\beta_i ) - r_i q(\beta_i)  \Big]^2
$$

And the new system has $2k +2$ variables. The degree is $2d$.

$$
\begin{split}
\tfrac{\partial}{\partial \beta } \mathcal{L} & = 0 \\
\tfrac{\partial}{\partial r } \mathcal{L}     & = 0 \\
\sum r_i -1 & = 0 \\
\sum \big[p(\beta_i) - r_i  q(\beta_i) \big]^2& = 0
\end{split}
$$

In [3]:
# Método 3 não funciona

n = 1000
k = 2
d = 4

a = exp( - 1 / sqrt(d) ) 
     
function N(b) 
    return prod(   [ b + a^k  for k in 1:d-1   ]  )   
end
    
function p(x)
    return x * ( N(x) - N(-x)) 
end 

function q(x)
    return  N(x) + N(-x)
end     

Random.seed!(126)

X = rand(n, k)
beta_0 = [2 , 0.1]


y = X* beta_0 + rand(Normal(0,1), n) ; 

@var β[1:k] r[1:k] λ μ


function Q(b)
    return  (y - X*b)' * (y - X*b)
end 


L = Q(β) + λ * (sum( [r[i] for i in 1:k ] ) - 1 ) +  μ*sum( [   ( p(β[i])  - r[i]*q(β[i]) )^2 for i in 1:k ] )


eqA = expand_vector(    [  differentiate( L , β[i])  for i in 1:k ]  ) 
eqB = expand_vector(    [  differentiate( L , r[i])  for i in 1:k ]  ) 
eqC = expand_vector(      sum([   r[i]  for i in 1:k ]) - 1  )
eqD = expand_vector(   sum( [   ( p(β[i])  - r[i]*q(β[i]) )^2 for i in 1:k ] )  )

F = vec( [eqA ; eqB ; eqC; eqD] ) 

F_sys = System(F; variables=[ β ; r ; λ; μ])

result = solve(F_sys; start_system=:total_degree) 

complex_sols_1 = solutions(result)

tol = 10^(-6)
display( count(u -> all(abs.(imag.(u)) .< tol ), complex_sols_1) ) 



Tracking 25088 paths... 100%|███████████████████████████| Time: 0:00:33
                   # paths tracked: 25088
   # non-singular solutions (real): 18 (0)
       # singular endpoints (real): 0 (0)
          # total solutions (real): 18 (0)


0

# Novas definições

Redefinindo o multiplicador, podemos escrever a derivada Lagrangeano do problema original é

$$
\begin{split}
\tfrac{\partial}{\partial \beta_i} Q_n(\beta)
+
\sum_{i=1}^k \frac{q(\beta_i) p'(\beta_i) - p(\beta_i)q'(\beta_i)}{ \lambda^2 q(\beta_i)^2} &  = 0 \\
\sum \frac{p ( \beta_i)}{q( \beta_i)} & = 1
\end{split}
$$

Usando $z_i = \frac{1}{\lambda q(x_i)}$

$$
\begin{split}
\tfrac{\partial}{\partial \beta_i} Q_n(\beta) + \sum z_i^2 \big[ q(\beta_i) p'(\beta_i) - p(\beta_i)q'(\beta_i) \big] & = 0 \quad (2d ) \\
\lambda z_i  q(\beta_i)   -1 & = 0 \quad (d) \\
\lambda \sum z_i p(\beta_i) -1 & = 0
\end{split}
$$




In [16]:

n = 1000
k = 2
d = 4

a = exp( - 1 / sqrt(d) ) 
     
function N(b) 
    return prod(   [ b + a^k  for k in 1:d-1   ]  )   
end
    
function p(x)
    return x * ( N(x) - N(-x)) 
end 

function q(x)
    return  N(x) + N(-x)
end     

Random.seed!(126)

X = rand(n, k)
beta_0 = [2 , 0.1]


y = X* beta_0 + rand(Normal(0,1), n) ; 

@var β[1:k] z[1:k] λ 


function Q(b)
    return  (y - X*b)' * (y - X*b)
end 


dp = [ differentiate( p(β[i]) ,   β[i])  for i in 1:k  ]
dq = [ differentiate( q(β[i])  ,  β[i])  for i in 1:k  ]
dQ = [ differentiate( Q(β) ,   β[i])  for i in 1:k  ]

# Sum = sum( [ z[i]^2 * ( q(β[i])*dp[i] - p(β[i])*dq[i]  )    for i in 1:k ]   )
Sum = sum( [ z[i] * ( q(β[i])*dp[i] - p(β[i])*dq[i]  )    for i in 1:k ]   )

eqA = expand_vector(    [  dQ[i] +  Sum    for i in 1:k ]  ) 
# eqB = expand_vector(    [  λ*z[i]*q(β[i])  - 1  for i in 1:k ]  ) 
# eqC = expand_vector(    λ * sum( [   z[i] * p(β[i])  for i in 1:k ]  ) - 1 ) 


eqB = expand_vector(    [  z[i]*q(β[i])^2 - λ  for i in 1:k ]  )
eqC = expand_vector(     sum( [    p(β[i])*q(β[i])*z[i] for i in 1:k ]  ) - λ )  

F = vec( [eqA ; eqB ; eqC] ) 

F_sys = System(F; variables=[ β ; z ; λ])

result = solve(F_sys; start_system=:total_degree) 

complex_sols_1 = solutions(result)

tol = 10^(-6)
display( count(u -> all(abs.(imag.(u)) .< tol ), complex_sols_1) ) 

S = [ real(s) for s in complex_sols_1 if all(abs.(imag.(s)) .< tol)] ; 

Tracking 6300 paths... 100%|████████████████████████████| Time: 0:00:04
                   # paths tracked: 6300
   # non-singular solutions (real): 11 (1)
       # singular endpoints (real): 0 (0)
          # total solutions (real): 11 (1)


1

In [17]:
S

1-element Vector{Vector{Float64}}:
 [2.0873271295076146, -0.002605440246281064, 2.61012178719941e-54, 4.276423536147513e-50, 3.3409558876152446e-52]

In [14]:
complex_sols_1

21-element Vector{Vector{ComplexF64}}:
 [1.3708147872253615 + 0.9203985024715323im, -0.9922352516261516 + 1.2712325282408132im, -1.8471823748012273 - 2.2444731834130733im, -2.798128382592682 - 1.6663672177699207im, 125.01776289609926 + 9.732337378491136im]
 [1.3708147872253615 - 0.9203985024715308im, -0.99223525162615 - 1.271232528240813im, -1.8471823748012246 + 2.2444731834130787im, -2.7981283825926835 + 1.6663672177699278im, 125.01776289609907 - 9.732337378490996im]
 [2.08921352184551 + 0.14762643149080373im, 3.9875996491357407e-19 + 0.2038981172125802im, 5.084306039769408e-21 + 1.7294782573848802e-20im, 0.5167884491961847 + 80.88958715047502im, 1.2951997987740492e-20 + 2.027293318245847e-18im]
 [2.088046111441513 + 0.14005075763067198im, -0.0016123994936781962 + 0.1934347765960074im, -3.9478778506128565e-5 - 6.164202104318253e-5im, -0.5318216484115877 - 80.84409784110285im, -0.0024584759404388794 - 0.007829384682712402im]
 [-1.0500077458017472e-18 + 0.2038981172125802im, -2.88557204